# Plotting Cubical Grids

This tutorial tours the plots for forests built with
`PersistenceForest.from_cubical_grid`: filtration snapshots, the image
background, cycle representatives, interior activity, Plotly views, 3D volumes
and animations. Each section ends with the knobs worth trying, so you can judge
the defaults and compare alternatives.

Two conventions matter throughout:

- **Axes.** By default array axis 0 runs along x and axis 1 along y, the frame
  in which cell orientations are defined. Pass `image_orientation=True` to draw
  rows downward and columns to the right, as an image viewer shows an array.
- **Grid conventions.** With `filtration_type="top_cells"` (the default) each
  entry is a pixel; with `"vertices"` each entry is a vertex and pixels take the
  maximum of their corners.

The executable cells do not save files by default. Set `SAVE_OUTPUTS = True` in
the animation section to write an MP4 under `examples/example_figures/`.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import HTML

from persforest import PersistenceForest
from persforest.measurements.cubical import cubical_chain_interior_volume, cubical_chain_volume


def ring_image(shape=(90, 120), seed=0):
    # Bright background with three dark rings; built with (row, column) axes.
    rng = np.random.default_rng(seed)
    rows, columns = np.mgrid[0:shape[0], 0:shape[1]] * 0.05
    rings = [((1.6, 1.8), 1.0), ((2.4, 4.3), 0.75), ((3.3, 2.4), 0.45)]
    distance = np.min([np.abs(np.hypot(rows - r, columns - c) - radius)
                       for (r, c), radius in rings], axis=0)
    return np.tanh(3 * distance) + 0.08 * rng.random(shape)


def shell_volume(n=18, seed=1):
    # A thick spherical shell and a small second shell touching it.
    rng = np.random.default_rng(seed)
    z, y, x = np.mgrid[0:n, 0:n, 0:n] / (n - 1) - 0.5
    big = np.abs(np.sqrt(x**2 + y**2 + z**2) - 0.3)
    small = np.abs(np.sqrt((x - 0.28)**2 + (y - 0.28)**2 + (z - 0.28)**2) - 0.12)
    return np.minimum(big, small) + 0.02 * rng.random(x.shape)

## Build a 2D forest

Low values enter the sublevel filtration first, so the dark rings close into
loops long before their bright interiors fill in. With `spacing=0.05`, cells are
placed in physical units.

In [ ]:
image = ring_image()
forest = PersistenceForest.from_cubical_grid(image, spacing=0.05, storage="cycles")
MIN_BAR = 0.1  # hides the short bars created by the pixel noise
long_bars = forest.longest_bars(3)
t_mid = float(np.median([bar.birth + 0.4 * bar.lifespan() for bar in long_bars]))
print(f"{len(forest.barcode)} bars; three longest:",
      [(round(bar.birth, 3), round(bar.death, 3)) for bar in long_bars])

fig, ax = plt.subplots(figsize=(6, 3))
forest.plot_barcode(ax=ax, min_bar_length=MIN_BAR, coloring="forest")

## Filtration snapshots and the axis convention

`plot_at_filtration` draws the image in grayscale, veils the pixels not yet
present and draws the active cycles along grid edges. On the left the default
physical frame (axis 0 along x), on the right `image_orientation=True`, which
matches how the array was built and how `plt.imshow(image)` shows it.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
forest.plot_at_filtration(t_mid, ax=axes[0], show=False, min_bar_length=MIN_BAR)
forest.plot_at_filtration(t_mid, ax=axes[1], show=False, min_bar_length=MIN_BAR, image_orientation=True)
axes[2].imshow(image, cmap="gray")
axes[2].set_title("plt.imshow(image) for reference")
fig.tight_layout()

## Image background

For 2D grids the image is shown by default: every grid value in grayscale
beneath the cycles, with a colorbar. It uses the `gray` colormap at
`image_alpha=0.75`, softened toward white so the bar colors stay dominant;
`style_2d` sets `image_colorbar=False` to drop the colorbar and
`colorbar_label` to rename it. Pixels not yet present are covered by a white
veil (`absent_veil_alpha=0.65`), so the sublevel set stays visible while
present pixels keep their true intensities. Pass `show_complex=False` for the
full image without the veil, or `show_image=False` to tint the present pixels
on a white background instead.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
forest.plot_at_filtration(t_mid, ax=axes[0], show=False, min_bar_length=MIN_BAR, image_orientation=True,
                          title="default: image with veil")
forest.plot_at_filtration(t_mid, ax=axes[1], show=False, min_bar_length=MIN_BAR, image_orientation=True,
                          show_complex=False, title="show_complex=False: full image")
forest.plot_at_filtration(t_mid, ax=axes[2], show=False, min_bar_length=MIN_BAR, image_orientation=True,
                          show_image=False, title="show_image=False: tinted complex")
fig.tight_layout()

### Compare image styles

`style_2d` sets `image_cmap`, `image_alpha`, `absent_veil_alpha` and
`cycle_edge_width`. Some alternatives side by side:

In [ ]:
variants = {
    "default: gray, alpha 0.75": {},
    "gray, alpha 0.5": {"image_alpha": 0.5},
    "Greys (dark = high)": {"image_cmap": "Greys"},
    "bone, alpha 0.6": {"image_cmap": "bone", "image_alpha": 0.6},
    "gray, wide cycles": {"cycle_edge_width": 3.0},
    "viridis, alpha 0.5": {"image_cmap": "viridis", "image_alpha": 0.5},
}
fig, axes = plt.subplots(2, 3, figsize=(15, 8.5))
for ax, (label, style) in zip(axes.flat, variants.items()):
    forest.plot_at_filtration(t_mid, ax=ax, show=False, min_bar_length=MIN_BAR, image_orientation=True,
                              show_image=True, show_complex=False, style_2d=style, title=label)
fig.tight_layout()

### Cycles through the filtration

The same view at several filtration values, with the veil showing how far the
sublevel set has grown.

In [ ]:
values = np.linspace(0.2, 1.0, 4)
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, value in zip(axes, values):
    forest.plot_at_filtration(value, ax=ax, show=False, min_bar_length=MIN_BAR,
                              image_orientation=True, show_image=True)
fig.tight_layout()

## Cycle representatives

`plot_barcode_cycle_reps` draws one representative per bar, sampled at
`birth + relative_position * lifespan`; longer bars are drawn on top of the
image.

In [ ]:
forest.plot_barcode_cycle_reps(min_bar_length=MIN_BAR, relative_position=0.2, figsize=(8, 5.5),
                               show=False, image_orientation=True, linewidth_cycle=2.0)

## Interior activity

`plot_interior_simplex_activity` colors every pixel by the bar whose interior
contains it; opacity grows with how long it stays inside. `overlap="layer"`
composites all bars instead of keeping the longest, and `show_complex` shades
the pixels present at `complex_max_filtration` underneath. `min_activity_length`
hides the short activity of noise loops.
`plot_interior_simplex_activity_gradient` blurs and blends the same per-pixel
activity into a smooth field.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
options = dict(show=False, dpi=100, image_orientation=True, min_activity_length=MIN_BAR)
forest.plot_interior_simplex_activity(ax=axes[0], **options)
forest.plot_interior_simplex_activity(ax=axes[1], overlap="layer", show_complex=True,
                                      complex_max_filtration=t_mid, **options)
forest.plot_interior_simplex_activity_gradient(ax=axes[2], **options)
for ax, label in zip(axes, ["longest", "layer + complex", "gradient"]):
    ax.set_title(label)
fig.tight_layout()

## Top cells versus vertices

With `filtration_type="vertices"` the values sit on vertices; the image
background draws one pixel per vertex, centered on it, so it overhangs the
cell domain by half a cell.

In [ ]:
vertex_forest = PersistenceForest.from_cubical_grid(image, spacing=0.05, filtration_type="vertices")
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
forest.plot_at_filtration(t_mid, ax=axes[0], show=False, min_bar_length=MIN_BAR, image_orientation=True,
                          show_image=True, title='filtration_type="top_cells"')
vertex_forest.plot_at_filtration(t_mid, ax=axes[1], show=False, min_bar_length=MIN_BAR,
                                 image_orientation=True, show_image=True,
                                 title='filtration_type="vertices"')
fig.tight_layout()

## Interactive Plotly views

`plot_at_filtration_plotly` and `plot_filtration_interactive` take the same
`image_orientation` and `show_image` options. The slider runs from the smallest
grid value to just past the last death. Hover over a cycle for its bar. In the
slider, the pixels present are outlined rather than shaded: frames then only
move lines, so the image stays still instead of redrawing at every step.

In [ ]:
fig = forest.plot_at_filtration_plotly(t_mid, min_bar_length=MIN_BAR, image_orientation=True,
                                       show_image=True, show=False, width=700, height=550)
fig

In [ ]:
fig = forest.plot_filtration_interactive(min_bar_length=MIN_BAR, resolution=40, image_orientation=True,
                                         show_image=True, show=False, width=750, height=600)
fig

## 3D volumes

For a volume the tracked homology is H2: cycles are closed surfaces of voxel
faces. `plot_at_filtration` draws the boundary surface of the voxels present
(light blue) and the active cycles; `style_3d` sets the camera and opacities.

In [ ]:
volume = shell_volume()
forest3d = PersistenceForest.from_cubical_grid(volume, storage="cycles")
bar3d = forest3d.max_bar()
t3d = bar3d.birth + 0.3 * bar3d.lifespan()
print(f"{len(forest3d.barcode)} bars; longest [{bar3d.birth:.3f}, {bar3d.death:.3f})")

fig = plt.figure(figsize=(14, 5))
ax = fig.add_subplot(1, 2, 1, projection="3d")
forest3d.plot_at_filtration(t3d, ax=ax, show=False, min_bar_length=0.01)
ax = fig.add_subplot(1, 2, 2, projection="3d")
forest3d.plot_at_filtration(t3d, ax=ax, show=False, min_bar_length=0.01, show_complex=False,
                            style_3d={"camera_eye": (35, -30), "cycle_face_alpha": 0.8},
                            title="cycles only, other camera")
fig.tight_layout()

In [ ]:
fig = plt.figure(figsize=(14, 5))
ax = fig.add_subplot(1, 2, 1, projection="3d")
forest3d.plot_interior_simplex_activity(ax=ax, show=False, dpi=100, geometry="cells")
ax.set_title('geometry="cells"')
ax = fig.add_subplot(1, 2, 2, projection="3d")
forest3d.plot_interior_simplex_activity(ax=ax, show=False, dpi=100, geometry="surface",
                                        show_complex=True, complex_max_filtration=0.03)
ax.set_title('geometry="surface", show_complex=True')
fig.tight_layout()

In [ ]:
fig = forest3d.plot_filtration_interactive(min_bar_length=0.01, resolution=25, show=False,
                                           width=750, height=600)
fig

In [ ]:
fig = forest3d.plot_interior_simplex_activity_plotly(edge_mode="surface", show=False,
                                                     width=700, height=550)
fig

## Animations

`animate_filtration` and `animate_barcode_measurement` work for grids too; the
label reads `t` (grid value) instead of the radius `r`, and the animation
starts at the smallest grid value. Options for `plot_at_filtration`, such as
`show_image`, go into `filtration_kwargs`.

In [ ]:
SAVE_OUTPUTS = False
OUTPUT_DIR = Path("../example_figures")

anim, anim_fig = forest.animate_filtration(
    frames=30,
    with_barcode=True,
    figsize=(10, 4),
    dpi=80,
    panel_width_ratios=(3.0, 2.0),
    panel_spacing=0.3,  # room for the colorbar of show_image
    filtration_kwargs={"show_image": True, "image_orientation": True, "min_bar_length": 0.05},
    barcode_kwargs={"min_bar_length": 0.05},
)
if SAVE_OUTPUTS:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    anim.save(OUTPUT_DIR / "cubical_filtration.mp4", fps=15, dpi=120)
plt.close(anim_fig)
HTML(anim.to_jshtml())

In [ ]:
anim, anim_fig = forest.animate_barcode_measurement(
    cubical_chain_interior_volume,
    bar=forest.max_bar(),
    frames=30,
    total_figsize=(10, 4),
    filtration_kwargs={"show_image": True, "image_orientation": True, "min_bar_length": 0.05},
)
plt.close(anim_fig)
HTML(anim.to_jshtml())

## Cycles and their measurement profiles

The three rings side by side with how a measurement evolves along each bar.
`compute_measurement_profiles` evaluates a measurement on every representative
of a bar, here the enclosed area, and returns one step function per bar. Each
profile uses its bar's color from the barcode and representatives.

In [ ]:
three = forest.longest_bars(3)
shortest = min(bar.lifespan() for bar in three)
profiles = forest.compute_measurement_profiles(cubical_chain_interior_volume, "enclosed area",
                                               min_bar_length=shortest)

fig, axes = plt.subplots(1, 3, figsize=(19, 4.8), gridspec_kw={"width_ratios": [1.25, 0.8, 1.1]})
forest.plot_barcode_cycle_reps(min_bar_length=shortest, relative_position=0.2, ax=axes[0], show=False,
                               image_orientation=True, linewidth_cycle=2.0, title="three longest bars")
forest.plot_barcode(ax=axes[1], min_bar_length=shortest, coloring="forest", bar_width=4)
for bar in sorted(three, key=lambda bar: bar.birth):
    profiles[bar].plot(ax=axes[2], color=forest.color_map_forest[bar], linewidth=2,
                       label=f"bar [{bar.birth:.2f}, {bar.death:.2f})")
axes[2].set_xlim(min(bar.birth for bar in three), max(bar.death for bar in three))
axes[2].set(xlabel="filtration value t", ylabel="enclosed area", title="measurement profiles")
axes[2].legend()
fig.tight_layout()

## Feedback checklist

Things worth judging while looking at the plots above:

1. Colorbar: placement and label of the intensity colorbar next to the image.
2. Default orientation: should `image_orientation=True` become the default for
   grids, given that most images are built with `(row, column)` axes?
3. Measurement profiles: is the three-panel view (cycles, barcode, profiles)
   the comparison you want, and is enclosed area the right default measurement?